# LoRA Integration with PyTorch Fully Sharded Data Parallel (FSDP)

## 1. Architectural Motivation & Theoretical Synergy
Fine-tuning billion-scale foundation models (e.g. LLaMA 70B, Qwen 72B) typically presents two challenges:
1. **Memory Ceiling**: Full fine-tuning requires 16-20 bytes per parameter (weights, gradients, optimizer states $m_t$ and $v_t$). A 70B model requires $\approx 1.2$ TB of VRAM.
2. **LoRA Parameter Efficiency**: Low-Rank Adaptation (LoRA) freezes the base parameters $W_0 \in \mathbb{R}^{d \times k}$ and trains low-rank decomposition matrices $A \in \mathbb{R}^{d \times r}$ and $B \in \mathbb{R}^{r \times k}$ ($r \ll d$). Only $\approx 0.1\%$ of parameters require gradient and optimizer state tracking.

### Combining FSDP with LoRA
When scaling beyond a single GPU, pairing **FSDP (Fully Sharded Data Parallel)** with **LoRA** provides extreme memory efficiency:
- Base weights ($W_0$) are sharded across $N$ GPUs and frozen (no optimizer states or gradients allocated).
- LoRA adapters ($A, B$) are sharded and updated.
- During forward pass, base weights are reconstructed via fast `All-Gather` and immediately freed after each layer.

$$\text{Total Memory per GPU} = \frac{\text{Frozen Weights}}{N} + \frac{\text{LoRA Weights} \times (1 + 1 + 2 \times 4)}{N} + \text{Activations}$$

In [ ]:
import numpy as np

class SimulatedFSDPLoRALayer:
    """
    Simulates a sharded Linear layer with LoRA adapter across N distributed GPUs under FSDP.
    Demonstrates zero-optimizer-memory base weights and sharded adapter updates.
    """
    def __init__(self, in_features, out_features, rank=8, num_shards=4):
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.num_shards = num_shards
        
        # Frozen base weight W0 [out_features, in_features]
        self.W0_full = np.random.randn(out_features, in_features) * 0.02
        # Shard base weight across ranks to save GPU memory
        self.W0_shards = np.split(self.W0_full, num_shards, axis=0)
        
        # LoRA Trainable Matrices: A [rank, in_features], B [out_features, rank]
        self.A = np.random.randn(rank, in_features) * 0.01
        self.B = np.zeros((out_features, rank)) # Initialize B to 0 for identity start
        self.alpha = 16.0
        self.scaling = self.alpha / self.rank

    def forward(self, x):
        """
        Simulates FSDP forward execution:
        1. All-Gather W0 shards to reconstruct full W0 in temporary buffer
        2. Compute base output: y_base = x @ W0.T
        3. Release gathered W0 buffer (freeing VRAM)
        4. Compute LoRA path: y_lora = (x @ A.T @ B.T) * scaling
        5. Return sum
        """
        # Step 1: All-Gather
        W0_gathered = np.concatenate(self.W0_shards, axis=0)
        
        # Step 2: Base Forward
        y_base = x @ W0_gathered.T
        
        # Step 3: Free buffer
        del W0_gathered
        
        # Step 4: LoRA adapter path
        y_lora = (x @ self.A.T @ self.B.T) * self.scaling
        
        return y_base + y_lora

# Verification
layer = SimulatedFSDPLoRALayer(in_features=128, out_features=256, rank=8, num_shards=4)
x = np.random.randn(2, 128)
output = layer.forward(x)

print(f"Input shape: {x.shape} | Output shape: {output.shape}")
print(f"Base Weight Size: {128 * 256} parameters (Frozen & Sharded)")
print(f"Trainable LoRA Size: {128*8 + 256*8} parameters ({((128*8 + 256*8)/(128*256))*100:.2f}% of base)")
print("Forward execution and buffer release executed successfully!")

## 2. Memory Consumption Analytical Model: Full Fine-Tuning vs FSDP + LoRA

In [ ]:
def calculate_training_memory(params_billions=70, lora_pct=0.002, gpus=8):
    """
    Calculates per-GPU memory consumption in GB for 70B parameter model.
    Assuming fp16 weights (2 bytes) and AdamW optimizer (8 bytes per trainable param).
    """
    total_params = params_billions * 1e9
    
    # 1. Full Fine-Tuning with FSDP
    # Sharded: Weights (2B), Gradients (2B), AdamW States (8B) -> Total 12 bytes/param
    full_fsdp_per_gpu = (total_params * 12) / gpus / (1024**3)
    
    # 2. LoRA with FSDP
    # Base weights: frozen (2 bytes/param sharded across GPUs)
    # LoRA params: (total_params * lora_pct) with Weights(2B) + Gradients(2B) + AdamW(8B)
    base_sharded_mem = (total_params * 2) / gpus / (1024**3)
    lora_params = total_params * lora_pct
    lora_mem = (lora_params * 12) / gpus / (1024**3)
    total_lora_fsdp = base_sharded_mem + lora_mem
    
    return full_fsdp_per_gpu, total_lora_fsdp

full_mem, lora_mem = calculate_training_memory(70, lora_pct=0.002, gpus=8)
print(f"LLaMA 70B Training on 8x A100 (80GB) GPUs:")
print(f"  Full FSDP Fine-Tuning Static Memory : {full_mem:.1f} GB / GPU (Requires Activation Offloading/OOM risk)")
print(f"  LoRA + FSDP Static Memory           : {lora_mem:.1f} GB / GPU (Leaves ~63 GB headroom for long sequences!)")

## 3. High-Yield Interview Questions & Best Practices

### Q1: What PyTorch FSDP wrapping policy must be used when fine-tuning with LoRA?
**Answer:** You must use a custom `auto_wrap_policy` or wrap layers using `transformer_auto_wrap_policy` targeting specific decoder blocks (e.g. `LlamaDecoderLayer`). If you wrap each individual linear layer separately, the communication overhead of `All-Gather` before every single matrix multiplication will severely degrade throughput. Wrapping at the Transformer Block level ensures that base weights for the entire block are gathered in one single network transfer.

### Q2: What is the gradient communication pattern when using LoRA with FSDP?
**Answer:** In full FSDP, every parameter requires a `Reduce-Scatter` during the backward pass to distribute gradients across ranks. With LoRA, because base model weights are frozen (`requires_grad=False`), **zero gradients are generated or communicated for base weights**. FSDP only executes `Reduce-Scatter` for the tiny LoRA adapter matrices ($A$ and $B$). This reduces interconnect communication volume by over **99%**, drastically accelerating training on ethernet or non-InfiniBand clusters.